# Tarea 3 — Aprendizaje supervisado: priorización de posibles falsas alarmas

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/ElenaVillano/databases/refs/heads/main/db_falsas_alarmas.csv"
df = pd.read_csv(url)


## Parte A: aprendizaje supervisado

### A1. Formulación y modelo inicial

In [2]:
df.head()

,folio,fecha_hora_creacion,fecha_hora_cierre,tipo_incidente_c4,incidente_c4,alcaldia_inicio,codigo_cierre,clas_con_f_alarma,tipo_entrada,alcaldia_cierre,alcaldia_catalogo,colonia_catalogo,longitud,latitud
0,C5/20220923/04016,2022-09-23 18:38:34,2022-09-23 22:22:27,Accidente,Choque sin lesionados,VENUSTIANO CARRANZA,A,EMERGENCIA,LLAMADA DEL 911,VENUSTIANO CARRANZA,Venustiano Carranza,20 De Noviembre,-99.108352,19.443920
1,C5/20220829/00630,2022-08-29 02:56:31,2022-08-29 06:04:53,Accidente,Motociclista,CUAUHTEMOC,A,URGENCIAS MEDICAS,LLAMADA DEL 911,CUAUHTEMOC,Cuauhtémoc,Obrera,-99.140843,19.416669
2,C5/20220724/01907,2022-07-24 04:29:09,2022-07-24 07:36:49,Accidente,Choque sin lesionados,IZTAPALAPA,A,EMERGENCIA,LLAMADA DEL 911,IZTAPALAPA,Iztapalapa,San Andres Tetepilco,-99.130702,19.371340
3,C5/20240215/03071,2024-02-15 16:20:03,2024-02-15 20:52:19,Accidente,Choque sin lesionados,LA MAGDALENA CONTRERAS,A,EMERGENCIA,LLAMADA DEL 911,LA MAGDALENA CONTRERAS,La Magdalena Contreras,La Cruz,-99.233973,19.305985
4,C5/20221006/03493,2022-10-06 19:33:05,2022-10-06 22:37:08,Accidente,Choque sin lesionados,IZTAPALAPA,F,EMERGENCIA,LLAMADA DEL 911,IZTAPALAPA,Iztapalapa,Santa Martha Acatitla Norte,-99.013140,19.374476


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 124261 entries, 0 to 124260
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   folio                124261 non-null  str    
 1   fecha_hora_creacion  124261 non-null  str    
 2   fecha_hora_cierre    124261 non-null  str    
 3   tipo_incidente_c4    124261 non-null  str    
 4   incidente_c4         124261 non-null  str    
 5   alcaldia_inicio      124253 non-null  str    
 6   codigo_cierre        124261 non-null  str    
 7   clas_con_f_alarma    124261 non-null  str    
 8   tipo_entrada         124261 non-null  str    
 9   alcaldia_cierre      124253 non-null  str    
 10  alcaldia_catalogo    124131 non-null  str    
 11  colonia_catalogo     121543 non-null  str    
 12  longitud             124261 non-null  float64
 13  latitud              124261 non-null  float64
dtypes: float64(2), str(12)
memory usage: 13.3 MB


In [10]:
df["codigo_cierre"].value_counts()

codigo_cierre
D    52963
A    45189
F    25219
I      890
Name: count, dtype: int64

In [12]:
df["codigo_cierre"].value_counts(normalize=True)

codigo_cierre
D    0.426224
A    0.363662
F    0.202952
I    0.007162
Name: proportion, dtype: float64

|Código|Significado|
|----|----|
|A|Afirmativo: la unidad llegó y confirmó la emergencia.|
|N|Negativo: la unidad llegó, pero nadie confirmó la emergencia o solicitó apoyo.|
|I|Informativo: solicitud de información.|
|F|Falso: el incidente fue considerado falso en el lugar.|
|D|Duplicado: se registró el mismo incidente más de una vez.|

**Unidad de predicción**
Cada fila representa una llamada o reporte individual, identificado mediante su folio.

**Momento de predicción**
El score se calcularía cuando se crea la llamada, antes de conocer cómo terminará o qué código de cierre recibirá.

**Etiqueta**
La etiqueta se construye a partir de codigo_cierre. La clase 1 corresponde a los reportes que cierran con F; la clase 0 corresponde a los que no cierran con F.

**Tratamiento de los códigos de cierre**
Aquí debes explicar tu decisión para A, D e I, y mencionar que N no aparece en esta base. También reconoce la limitación de D: un duplicado no necesariamente revela si el incidente era real, pero al llegar la llamada el C5 aún no sabe que será duplicada.

**Tasa base**
Es aproximadamente 20.3%, es decir, la proporción de reportes que cierran con F en la base, si conservan los demás códigos como clase 0.

| Feature | Tipo | Disponible al momento del reporte |
|---------|------|:-------------------------------:|
| `incidente_c4` | Categórica | Sí |
| `tipo_incidente_c4` | Categórica | Sí |
| `alcaldia_inicio` | Categórica | Sí |
| `tipo_entrada` | Categórica | Sí |
| `hora_del_dia` | Numérica | Sí |
| `dia_semana` | Numérica | Sí |

Los predictores disponibles son: fecha_hora_creacion, tipo_incidente_c4, incidente_c4, alcaldia_inicio, tipo_entrada. Por su nombre, describen información registrada al inicio de la llamada.

Usaremos el modelo de dummy, esperando que al tener el mismo score, los 100 reportes selecionados den una eficiecia aproximamda del 20.3%. 

**Limpieza y Pocesamiento**

In [16]:
df['fecha_hora_creacion'] = pd.to_datetime(df['fecha_hora_creacion'])
df['fecha_hora_cierre'] = pd.to_datetime(df['fecha_hora_cierre'])

print(df['fecha_hora_creacion'].min(), df['fecha_hora_creacion'].max())

2021-12-31 20:59:59 2024-02-29 20:53:56


In [17]:
df['y_falsa_alarma'] = (df['codigo_cierre'] == 'F').astype(int)

df['y_falsa_alarma'].mean()

np.float64(0.20295185134515253)

Primero pásamos de texto a datetime para extraer fecha y hora como predictores y par agrupar las variables. 